In [ ]:
# ==============================================================================
# CELL 1: MAIN MILESTONE 3 PIPELINE (GTSRB + EVALUATION + LANE + OVERLAY)
# ==============================================================================

import os
import cv2
import json
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Preprocessing
class CLAHETransform:
    """Applies CLAHE on LAB L-channel to recover contrast without washing color."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8), tileGridSize=None):
        self.clip_limit = clip_limit
        self.tile_grid_size = tileGridSize if tileGridSize is not None else tile_grid_size

    def __call__(self, pil_img):
        img_rgb = np.array(pil_img)
        lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        img_out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        return torchvision.transforms.functional.to_pil_image(img_out)

IMG_SIZE = 96
NUM_CLASSES = 43
BATCH_SIZE = 64

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Load GTSRB Dataset (with fallback)
class SyntheticSignDataset(Dataset):
    def __init__(self, length, transform=None):
        self.length = length
        self.transform = transform

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        cls_id = idx % NUM_CLASSES
        img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        cv2.circle(img, (IMG_SIZE // 2, IMG_SIZE // 2), IMG_SIZE // 3, (20 + cls_id * 4, 60, 180), -1)
        cv2.putText(img, str(cls_id), (IMG_SIZE // 3, IMG_SIZE // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        pil_img = torchvision.transforms.functional.to_pil_image(img)
        if self.transform:
            pil_img = self.transform(pil_img)
        return pil_img, cls_id

try:
    train_dataset = datasets.GTSRB(root="./data", split="train", transform=train_transform, download=True)
    test_dataset = datasets.GTSRB(root="./data", split="test", transform=eval_transform, download=True)
    print(f"Loaded real GTSRB -> Train: {len(train_dataset)}, Test: {len(test_dataset)}")
except Exception as e:
    print(f"GTSRB download fallback triggered ({e}). Using synthetic fallback.")
    train_dataset = SyntheticSignDataset(860, transform=train_transform)
    test_dataset = SyntheticSignDataset(215, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# 3. Model Builders & Training
def build_model(arch="mobilenet_v2", num_classes=43, freeze_backbone=False):
    if arch == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        if freeze_backbone:
            for p in model.features.parameters():
                p.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif arch == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze_backbone:
            for name, p in model.named_parameters():
                if not name.startswith("fc."):
                    p.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f"Unsupported architecture: {arch}")
    return model.to(device)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_and_eval(model, epochs=3, lr=5e-4, name="Model"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    history = {"train_acc": [], "test_acc": [], "loss": []}
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        correct_train, total_train, running_loss = 0, 0, 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()    #clears gradients no mixup
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            correct_train += (out.argmax(dim=1) == y).sum().item()
            total_train += y.size(0)

        train_acc = 100.0 * correct_train / total_train

        model.eval()
        correct_test, total_test = 0, 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                correct_test += (model(x).argmax(dim=1) == y).sum().item()
                total_test += y.size(0)

        test_acc = 100.0 * correct_test / total_test
        avg_loss = running_loss / max(total_train, 1)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        history["loss"].append(avg_loss)
        print(f"[{name}] Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    elapsed = time.time() - start_time
    print(f"[{name}] Finished in {elapsed:.1f}s | Trainable Params: {count_params(model):,}\n")
    return history, elapsed

EPOCHS = 3
mobilenet = build_model("mobilenet_v2", NUM_CLASSES, freeze_backbone=False)
resnet = build_model("resnet18", NUM_CLASSES, freeze_backbone=False)

m_hist, m_time = train_and_eval(mobilenet, epochs=EPOCHS, lr=5e-4, name="MobileNetV2")
r_hist, r_time = train_and_eval(resnet, epochs=EPOCHS, lr=5e-4, name="ResNet18")

summary_df = pd.DataFrame([
    {
        "Model": "MobileNetV2",
        "Trainable Params": f"{count_params(mobilenet):,}",
        "Final Train Acc (%)": f"{m_hist['train_acc'][-1]:.2f}",
        "Final Test Acc (%)": f"{m_hist['test_acc'][-1]:.2f}",
        "Train Time (s)": f"{m_time:.1f}"
    },
    {
        "Model": "ResNet18",
        "Trainable Params": f"{count_params(resnet):,}",
        "Final Train Acc (%)": f"{r_hist['train_acc'][-1]:.2f}",
        "Final Test Acc (%)": f"{r_hist['test_acc'][-1]:.2f}",
        "Train Time (s)": f"{r_time:.1f}"
    }
])
print("=== MODEL COMPARISON SUMMARY ===")
print(summary_df.to_string(index=False))

plt.figure(figsize=(10, 4))
plt.plot(range(1, EPOCHS + 1), m_hist["train_acc"], marker="o", label="MobileNetV2 Train Acc")
plt.plot(range(1, EPOCHS + 1), m_hist["test_acc"], marker="s", label="MobileNetV2 Test Acc")
plt.plot(range(1, EPOCHS + 1), r_hist["test_acc"], marker="^", linestyle="--", label="ResNet18 Test Acc")
plt.title("Milestone 3: Accuracy Progression Across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("training_curve.png", dpi=150)
plt.show()

mobilenet.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds = mobilenet(x.to(device)).argmax(dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(y.numpy())

print("\n=== CLASSIFICATION REPORT (MobileNetV2) ===")
print(classification_report(y_true, y_pred, zero_division=0, digits=3))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues", cbar=True)
plt.title("MobileNetV2 Confusion Matrix (43 Classes)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# 4. Lane Detector
class TemporalLaneDetector:
    def __init__(self, alpha=0.7):
        self.alpha = alpha
        self.prev_left = None
        self.prev_right = None

    def roi_mask(self, edges):
        h, w = edges.shape[:2]
        poly = np.array([[
            (int(w * 0.10), h),
            (int(w * 0.44), int(h * 0.62)),
            (int(w * 0.56), int(h * 0.62)),
            (int(w * 0.90), h)
        ]], dtype=np.int32)
        mask = np.zeros_like(edges)
        cv2.fillPoly(mask, poly, 255)
        return cv2.bitwise_and(edges, mask)

    def _fit_line(self, pts, y_min, y_max):
        if len(pts) < 2:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        poly = np.polyfit(ys, xs, deg=1)
        x1 = int(np.polyval(poly, y_max))
        x2 = int(np.polyval(poly, y_min))
        return (x1, int(y_max), x2, int(y_min))

    def _smooth(self, curr, prev):
        if curr is None:
            return prev
        if prev is None:
            return curr
        return tuple(int(self.alpha * c + (1 - self.alpha) * p) for c, p in zip(curr, prev))

    def detect(self, frame_bgr):
        h, w = frame_bgr.shape[:2]
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        lab = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l_eq = clahe.apply(lab[:, :, 0])
        combined = cv2.addWeighted(gray, 0.5, l_eq, 0.5, 0)

        blur = cv2.GaussianBlur(combined, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)
        masked = self.roi_mask(edges)

        lines = cv2.HoughLinesP(
            masked, rho=1, theta=np.pi / 180,
            threshold=30, minLineLength=30, maxLineGap=40
        )

        left_pts, right_pts = [], []
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                if x2 == x1:
                    continue
                slope = (y2 - y1) / (x2 - x1)
                if abs(slope) < 0.35:
                    continue
                if slope < 0 and x1 < w * 0.55 and x2 < w * 0.55:
                    left_pts.extend([(x1, y1), (x2, y2)])
                elif slope > 0 and x1 > w * 0.45 and x2 > w * 0.45:
                    right_pts.extend([(x1, y1), (x2, y2)])

        y_max = h
        y_min = int(h * 0.63)
        left_line = self._smooth(self._fit_line(left_pts, y_min, y_max), self.prev_left)
        right_line = self._smooth(self._fit_line(right_pts, y_min, y_max), self.prev_right)
        self.prev_left = left_line
        self.prev_right = right_line
        return left_line, right_line

# 5. Baseline Overlay + FPS Benchmark
tusimple_json_files = glob.glob("./**/label_data_*.json", recursive=True)
frames_bgr = []
if tusimple_json_files:
    base_dir = os.path.dirname(tusimple_json_files[0])
    with open(tusimple_json_files[0], "r") as f:
        for idx, line in enumerate(f):
            if idx >= 15:
                break
            rec = json.loads(line)
            img_path = os.path.join(base_dir, rec["raw_file"])
            if os.path.exists(img_path):
                img = cv2.imread(img_path)
                if img is not None:
                    frames_bgr.append(cv2.resize(img, (640, 360)))

if not frames_bgr:
    for t in range(15):
        frame = np.zeros((360, 640, 3), dtype=np.uint8)
        cv2.rectangle(frame, (0, 180), (640, 360), (65, 65, 65), -1)
        shift = int(8 * np.sin(t * 0.4))
        cv2.line(frame, (120 + shift, 360), (285, 215), (255, 255, 255), 7)
        cv2.line(frame, (520 + shift, 360), (355, 215), (0, 220, 255), 7)
        frames_bgr.append(frame)

lane_detector = TemporalLaneDetector(alpha=0.75)
sample_sign_pil, _ = test_dataset[0]
mobilenet.eval()
with torch.no_grad():
    pred_class = mobilenet(sample_sign_pil.unsqueeze(0).to(device)).argmax(dim=1).item()

fps_list = []
last_out = None
for f_bgr in frames_bgr:
    t0 = time.time()
    left_l, right_l = lane_detector.detect(f_bgr)
    out_frame = f_bgr.copy()
    if left_l is not None:
        cv2.line(out_frame, (left_l[0], left_l[1]), (left_l[2], left_l[3]), (0, 255, 0), 4)
    if right_l is not None:
        cv2.line(out_frame, (right_l[0], right_l[1]), (right_l[2], right_l[3]), (0, 255, 0), 4)
    cv2.rectangle(out_frame, (20, 20), (290, 65), (25, 25, 25), -1)
    cv2.putText(out_frame, f"Sign Class ID: {pred_class}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2)
    fps = 1.0 / max(time.time() - t0, 1e-6)
    fps_list.append(fps)
    cv2.putText(out_frame, f"FPS: {fps:.1f}", (500, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (50, 255, 50), 2)
    last_out = out_frame

print(f"Average Overlay Processing Speed: {np.mean(fps_list):.2f} FPS")
cv2.imwrite("adas_overlay_demo.jpg", last_out)
plt.figure(figsize=(10, 5))
plt.imshow(cv2.cvtColor(last_out, cv2.COLOR_BGR2RGB))
plt.title(f"Baseline ADAS Overlay (Avg FPS: {np.mean(fps_list):.1f})")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELL 2: BENCHMARK REAL GTSRB GRID + CUSTOM STILL IMAGES + DEFAULT VIDEO CHECK
# ==============================================================================

import os
import cv2
import time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image

GTSRB_NAMES = {
    0: "Speed limit (20km/h)", 1: "Speed limit (30km/h)", 2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)", 4: "Speed limit (70km/h)", 5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)", 7: "Speed limit (100km/h)", 8: "Speed limit (120km/h)",
    9: "No passing", 10: "No passing veh over 3.5t", 11: "Right-of-way at intersection",
    12: "Priority road", 13: "Yield", 14: "Stop", 15: "No vehicles",
    16: "Veh > 3.5t prohibited", 17: "No entry", 18: "General caution",
    19: "Dangerous curve left", 20: "Dangerous curve right", 21: "Double curve",
    22: "Bumpy road", 23: "Slippery road", 24: "Road narrows on right",
    25: "Road work", 26: "Traffic signals", 27: "Pedestrians", 28: "Children crossing",
    29: "Bicycles crossing", 30: "Ice/snow", 31: "Wild animals crossing",
    32: "End speed + passing limits", 33: "Turn right ahead", 34: "Turn left ahead",
    35: "Ahead only", 36: "Go straight or right", 37: "Go straight or left",
    38: "Keep right", 39: "Keep left", 40: "Roundabout mandatory",
    41: "End of no passing", 42: "End no passing veh > 3.5t"
}

inv_normalize = transforms.Normalize(
    mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
    std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
)

def predict_sign_with_conf(model, tensor_chw):
    model.eval()
    with torch.no_grad():
        logits = model(tensor_chw.unsqueeze(0).to(device))
        probs = F.softmax(logits, dim=1)[0]
        conf, idx = torch.max(probs, dim=0)
    return int(idx.item()), float(conf.item()) * 100.0

def draw_adas_hud(frame_bgr, left_line, right_line, sign_id, sign_conf, fps, sign_crop_bgr=None):
    out = frame_bgr.copy()
    h, w = out.shape[:2]

    if left_line is not None and right_line is not None:
        lane_poly = np.array([[
            (left_line[0], left_line[1]),
            (left_line[2], left_line[3]),
            (right_line[2], right_line[3]),
            (right_line[0], right_line[1])
        ]], dtype=np.int32)
        overlay_poly = out.copy()
        cv2.fillPoly(overlay_poly, lane_poly, (0, 200, 0))
        out = cv2.addWeighted(overlay_poly, 0.22, out, 0.78, 0)

    if left_line is not None:
        cv2.line(out, (left_line[0], left_line[1]), (left_line[2], left_line[3]), (0, 255, 0), 4)
    if right_line is not None:
        cv2.line(out, (right_line[0], right_line[1]), (right_line[2], right_line[3]), (0, 255, 255), 4)

    label_name = GTSRB_NAMES.get(sign_id, f"Class {sign_id}")
    cv2.rectangle(out, (15, 15), (430, 82), (20, 20, 20), -1)
    cv2.rectangle(out, (15, 15), (430, 82), (0, 200, 255), 2)
    cv2.putText(out, f"SIGN [{sign_id}]: {label_name[:28]}", (25, 43),
                cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 2)
    cv2.putText(out, f"Confidence: {sign_conf:.1f}%", (25, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.58, (100, 255, 180), 2)

    if sign_crop_bgr is not None:
        thumb = cv2.resize(sign_crop_bgr, (64, 64))
        x0, y0 = 440, 16
        if y0 + 64 <= h and x0 + 64 <= w:
            out[y0:y0+64, x0:x0+64] = thumb
            cv2.rectangle(out, (x0, y0), (x0+64, y0+64), (255, 255, 255), 2)

    cv2.rectangle(out, (w - 145, 15), (w - 15, 55), (20, 20, 20), -1)
    cv2.putText(out, f"FPS: {fps:.1f}", (w - 132, 43),
                cv2.FONT_HERSHEY_SIMPLEX, 0.72, (0, 255, 0), 2)
    return out

# ------------------------------------------------------------------------------
# 1. REAL GTSRB 2x4 BENCHMARK GRID
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
sample_indices = np.linspace(0, len(test_dataset) - 1, 8, dtype=int)

for ax, idx in zip(axes.flat, sample_indices):
    img_t, true_id = test_dataset[int(idx)]
    pred_id, conf = predict_sign_with_conf(mobilenet, img_t)
    vis_t = torch.clamp(inv_normalize(img_t.clone()), 0.0, 1.0)
    vis_np = (vis_t.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    ax.imshow(vis_np)
    color = "green" if pred_id == true_id else "red"
    ax.set_title(f"True: {true_id} | Pred: {pred_id} ({conf:.0f}%)", fontsize=9, color=color, fontweight="bold")
    ax.axis("off")

plt.suptitle("Real GTSRB Test Images — MobileNetV2 Predictions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("real_sign_predictions.png", dpi=150)
plt.show()

# ------------------------------------------------------------------------------
# 2. CUSTOM INPUT IMAGES (SIGN CLASSIFICATION + ROAD LANE OVERLAY)
# ------------------------------------------------------------------------------
CUSTOM_SIGN_PATHS = ["paule-knete-qKKjDFPCts8-unsplash.jpg", "jaykumar-bherwani-o9T6G8IWTh4-unsplash.jpg","joshua-hoehne-U4rGvsvop-s-unsplash.jpg"]
CUSTOM_ROAD_PATHS = ["pexels-iriser-15560907.jpg","kevin-schmid-ldQh3wZNW5M-unsplash.jpg"]

valid_signs = [p for p in CUSTOM_SIGN_PATHS if os.path.exists(p)]
if valid_signs:
    fig, axes = plt.subplots(1, len(valid_signs), figsize=(4 * len(valid_signs), 4))
    if len(valid_signs) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, valid_signs):
        pil_img = Image.open(img_path).convert("RGB")
        img_t = eval_transform(pil_img)
        pred_id, conf = predict_sign_with_conf(mobilenet, img_t)
        sign_name = GTSRB_NAMES.get(pred_id, f"Class {pred_id}")

        ax.imshow(pil_img)
        ax.set_title(f"{os.path.basename(img_path)}\nPred [{pred_id}]: {sign_name[:22]} ({conf:.1f}%)",
                     fontsize=9, fontweight="bold", color="blue")
        ax.axis("off")
    plt.suptitle("Custom Input Sign Images — MobileNetV2 Predictions", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("custom_sign_predictions.png", dpi=150)
    plt.show()
else:
    print("No custom sign images found yet; add filenames to CUSTOM_SIGN_PATHS if desired.")

sign_t, _ = test_dataset[14]
sign_id, sign_conf = predict_sign_with_conf(mobilenet, sign_t)
sign_vis_rgb = (torch.clamp(inv_normalize(sign_t.clone()), 0.0, 1.0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
sign_vis_bgr = cv2.cvtColor(sign_vis_rgb, cv2.COLOR_RGB2BGR)

valid_roads = [p for p in CUSTOM_ROAD_PATHS if os.path.exists(p)]
if valid_roads:
    if valid_signs:
        first_pil = Image.open(valid_signs[0]).convert("RGB")
        hud_sign_id, hud_sign_conf = predict_sign_with_conf(mobilenet, eval_transform(first_pil))
        hud_sign_bgr = cv2.cvtColor(np.array(first_pil), cv2.COLOR_RGB2BGR)
    else:
        hud_sign_id, hud_sign_conf, hud_sign_bgr = sign_id, sign_conf, sign_vis_bgr

    for idx, road_path in enumerate(valid_roads):
        road_bgr = cv2.imread(road_path)
        if road_bgr is None:
            continue
        resized = cv2.resize(road_bgr, (640, 360))
        t0 = time.time()
        img_lane_detector = TemporalLaneDetector(alpha=1.0)
        left_l, right_l = img_lane_detector.detect(resized)
        fps_img = 1.0 / max(time.time() - t0, 1e-6)

        hud_frame = draw_adas_hud(resized, left_l, right_l, hud_sign_id, hud_sign_conf, fps_img, hud_sign_bgr)
        out_name = f"custom_road_hud_{idx + 1}.jpg"
        cv2.imwrite(out_name, hud_frame)

        plt.figure(figsize=(10, 5))
        plt.imshow(cv2.cvtColor(hud_frame, cv2.COLOR_BGR2RGB))
        plt.title(f"Custom Road Image ADAS Overlay ({os.path.basename(road_path)})")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("No custom road images found yet; add filenames to CUSTOM_ROAD_PATHS if desired.")

# ------------------------------------------------------------------------------
# 3. DEFAULT VIDEO CHECK (OPTIONAL)
# ------------------------------------------------------------------------------
VIDEO_PATH = None
for candidate in ["dashcam.mp4", "road.mp4", "video.mp4", "sample_road.mp4"]:
    if os.path.exists(candidate):
        VIDEO_PATH = candidate
        break

real_lane_detector = TemporalLaneDetector(alpha=0.75)

if VIDEO_PATH is not None:
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps_input = cap.get(cv2.CAP_PROP_FPS)
    out_fps = fps_input if fps_input and fps_input > 1 else 20.0
    out_w, out_h = 640, 360
    writer = cv2.VideoWriter("adas_real_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), out_fps, (out_w, out_h))

    fps_vals, preview_frames, frame_count = [], [], 0
    while cap.isOpened() and frame_count < 300:
        ret, frame = cap.read()
        if not ret:
            break
        resized = cv2.resize(frame, (out_w, out_h))
        t0 = time.time()
        left_l, right_l = real_lane_detector.detect(resized)
        hud_frame = draw_adas_hud(resized, left_l, right_l, sign_id, sign_conf, 0.0, sign_vis_bgr)
        cur_fps = 1.0 / max(time.time() - t0, 1e-6)
        fps_vals.append(cur_fps)
        cv2.rectangle(hud_frame, (out_w - 145, 15), (out_w - 15, 55), (20, 20, 20), -1)
        cv2.putText(hud_frame, f"FPS: {cur_fps:.1f}", (out_w - 132, 43),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.72, (0, 255, 0), 2)
        writer.write(hud_frame)
        if frame_count in [0, 20, 40]:
            preview_frames.append(cv2.cvtColor(hud_frame, cv2.COLOR_BGR2RGB))
        frame_count += 1

    cap.release()
    writer.release()
    print(f"Processed {frame_count} frames | Avg FPS: {np.mean(fps_vals):.1f} -> Saved adas_real_output.mp4")
    if preview_frames:
        plt.figure(figsize=(12, 4))
        plt.imshow(preview_frames[-1])
        plt.title(f"Real Video ADAS Output ({VIDEO_PATH})")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig("real_video_frame.png", dpi=150)
        plt.show()
else:
    print("No default video file found in Cell 2.")